# 00 — Setup de Sentry

Inicializa el entorno de trabajo en Google Colab.

Ejecutar todas las celdas de arriba abajo cada vez que abras una nueva sesión.



## 1. Clonar el repositorio desde GitHub

Este bloque limpia cualquier clonado previo y clona de nuevo desde GitHub.

**Patrón a prueba de errores:**
1. Borra `/content/sentry` si existe.
2. Va a `/content` (raíz).
3. Clona.
4. Entra al repositorio.
5. Verifica.

In [19]:
import os

print(f"Directorio actual: {os.getcwd()}")

if os.path.exists("/content/sentry"):
    print("Borrando /content/sentry existente...")
    !rm -rf /content/sentry

%cd /content
!git clone https://github.com/AguCS231/sentry.git
%cd /content/sentry

print("\n=== Verificación ===")
!pwd
!ls -la

Directorio actual: /content/sentry
Borrando /content/sentry existente...
/content
Cloning into 'sentry'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/sentry

=== Verificación ===
/content/sentry
total 16
drwxr-xr-x 3 root root 4096 Sep 22 23:41 .
drwxr-xr-x 1 root root 4096 Sep 22 23:41 ..
drwxr-xr-x 8 root root 4096 Sep 22 23:41 .git
-rw-r--r-- 1 root root  151 Sep 22 23:41 README.md


## 2. Configurar Git

Git necesita saber quién eres para firmar los commits. Esta configuración se borra cuando Colab se reinicia, así que hay que hacerlo cada vez.

In [20]:
!git config --global user.email "srcasares2@gmail.com"
!git config --global user.name "AguCS231"
!git config --list | grep user

user.email=srcasares2@gmail.com
user.name=AguCS231


## 3. Verificar el estado del repositorio

Comprobamos que estamos sincronizados con GitHub. Debería salir `nothing to commit, working tree clean`.

In [21]:
!git status
!git log --oneline -5

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
903b52d (HEAD -> main, origin/main, origin/HEAD) Initial commit


## 4. Instalar dependencias

Solo la primera vez por sesión. Tarda 2-4 minutos.

**Versiones fijas** para garantizar la reproducibilidad del proyecto.

In [22]:
!pip install pyspark==3.5.0 pandas numpy scikit-learn matplotlib seaborn -q

## 5. Verificar versiones

Comprobamos que todas las librerías están instaladas con la versión correcta.

In [23]:
import pyspark, pandas, numpy, sklearn, matplotlib, seaborn

print("=== Versiones ===")
print(f"PySpark:       {pyspark.__version__}")
print(f"Pandas:        {pandas.__version__}")
print(f"NumPy:         {numpy.__version__}")
print(f"scikit-learn:  {sklearn.__version__}")
print(f"matplotlib:    {matplotlib.__version__}")
print(f"seaborn:       {seaborn.__version__}")

=== Versiones ===
PySpark:       3.5.0
Pandas:        2.2.3
NumPy:         2.1.3
scikit-learn:  1.6.1
matplotlib:    3.10.0
seaborn:       0.13.2


## 6. Crear una sesión de Spark

Comprobamos que PySpark funciona correctamente.

In [24]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("VIGIA_Setup") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Sesión de Spark creada: {spark.version}")

# Prueba rápida
data = [("BENIGN", 1000), ("DDoS", 500), ("PortScan", 200)]
df = spark.createDataFrame(data, ["Label", "Count"])
df.show()

# Cerrar la sesión (buena práctica)
spark.stop()

Sesión de Spark creada: 3.5.0
+--------+-----+
|   Label|Count|
+--------+-----+
|  BENIGN| 1000|
|    DDoS|  500|
|PortScan|  200|
+--------+-----+



## 7. Checklist final

Si todo lo anterior se ha ejecutado sin errores, el entorno está listo.

- [ ] Repo clonado en `/content/sentry`
- [ ] Git configurado con tu email y nombre
- [ ] `git status` sin cambios pendientes
- [ ] Librerías instaladas
- [ ] Versiones verificadas
- [ ] Spark funciona

**Siguiente paso:** abrir `notebooks/01_EDA_CICIDS2017.ipynb` para empezar el análisis.

## Solución de problemas

### `fatal: not a git repository`
No estás dentro del repo. Ejecuta `%cd /content/sentry` y vuelve a intentar.

### `fatal: destination path 'sentry' already exists`
Ya hay un clonado. La Celda 3 lo borra automáticamente.

### `metadata-generation-failed` al instalar
Probablemente estás forzando una versión incompatible con Colab. Usa las versiones de la Celda 9 tal cual.

### El push pide usuario y contraseña
La contraseña es un **token** de GitHub, no tu contraseña de la cuenta.
Genera uno en: GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic). Necesita el scope `repo`.